Contexto

O setor de fraudes apontou que existem clientes que se queixaram de não contratar serviços específicos, como o crédito pessoal. No entanto, os protocolos de segurança da senha foram realizados em conformidade — cada cliente autenticou com sua própria senha. Em função disso, o banco precisa arcar com reembolsos e medidas de contenção para evitar processos judiciais, pois os clientes alegam terem sido invadidos por hackers ou algo similar.

Para reforçar a segurança, o banco quer implementar uma autenticação facial complementar à senha, a depender do serviço (ex.: crédito pessoal). Caso o cliente não seja autenticado, ele deve ser direcionado a uma esteira dedicada de atendimento, e as evidências da não identificação devem ser encaminhadas à área de IA para validação de parâmetros e limiares, visando o aperfeiçoamento do modelo.

O que deve ser construído (obrigatório)

1. Detector de face — localizar e recortar o(s) rosto(s) presentes em uma imagem ou frame de vídeo.
2. Identificação facial — comparar o rosto capturado com uma base de clientes cadastrados e decidir se há correspondência (match/no match), com um limiar de decisão explícito.
3. Detecção de vivacidade (liveness) — comprovar que o rosto pertence a uma pessoa real no momento da captura, impedindo que um fraudador utilize uma foto estática ou imagem de tela/impressa.

O fluxo final deve simular o caso de negócio: cliente autenticado → acesso liberado; cliente não autenticado (falha de identificação ou de liveness) → encaminhado para esteira de exceção, com evidência registrada (imagem e/ou metadados) para futura análise da equipe de IA.

Entregáveis

- Código-fonte: notebook (`.ipynb`) ou script (`.py`), completo e executável do início ao fim, com comentários explicando cada módulo (detecção, identificação, liveness e a integração entre eles).
- Vídeo explicativo: demonstração da aplicação em execução, mostrando pelo menos (a) um caso de autenticação aprovada e (b) um caso de rejeição (identificação ou liveness falhando). Envie o vídeo pelo sistema ou compartilhe um link (Google Drive ou YouTube).

Observações importantes sobre a entrega

- O link do vídeo deve apontar para o arquivo direto, não para uma pasta, e precisa estar com permissão de visualização aberta ("qualquer pessoa com o link").
- O dataset de rostos usado para cadastro/teste deve ser composto por fotos do próprio grupo (ou de pessoas que deram consentimento explícito) — não utilizem fotos de terceiros ou de pessoas públicas sem autorização.
- O notebook/script deve rodar de ponta a ponta sem erros

# Detecção e Reconhecimento Facial

FIAP · MBA Data Science & Artificial Intelligence — Visão Computacional · Trabalho Final

## 1. Requerimentos

Todas as bibliotecas já estão instaladas no Google Colab.

* OpenCV >= 3.4.3
* Matplotlib >= 3.1.3
* Seaborn >= 0.0.10
* Numpy >= 1.18.1
* Mediapipe==0.10.9

### 1.2 Arquivos

Baixe o repositório do GitHub utilizando o comando abaixo. Em caso de atualização, utilize o comando para apagar o diretório antes.

Vamos agora posicionar o diretório do repositório para a aula respectiva. Nesse caso envie o comando de mudança de diretório.

Importação das bibliotecas.

In [91]:
%cd content

[Errno 2] No such file or directory: 'content'
/content/computer-vision/computer-vision/computer-vision/computer-vision


In [92]:
!rm -rf "computer-vision/"

In [93]:
!git clone https://github.com/VirtuousDeath/computer-vision.git


Cloning into 'computer-vision'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 10 (delta 0), reused 2 (delta 0), pack-reused 7 (from 1)
Receiving objects: 100% (10/10), 107.59 MiB | 25.14 MiB/s, done.


In [94]:
%cd computer-vision

/content/computer-vision/computer-vision/computer-vision/computer-vision/computer-vision


In [95]:
!apt-get install -y libopencv-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopencv-dev is already the newest version (4.6.0+dfsg-13.1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [96]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-contrib-python==5.0.0.93

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-contrib-python 5.0.0.93
Uninstalling opencv-contrib-python-5.0.0.93:
  Successfully uninstalled opencv-contrib-python-5.0.0.93
  Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (82.1 MB)


# OpenCV versão 5

No OpenCV 5.0.0, o tradicional cv2.CascadeClassifier foi oficialmente removido da biblioteca principal. Como parte de uma grande reformulação para tornar o mecanismo mais enxuto e moderno, o OpenCV moveu todos os detectores de objetos legados baseados em Haar-cascade e HOG do módulo principal e os realocou para o repositório opencv_contrib, sob o módulo xobjdetect.

In [97]:
!yes A | unzip faces.zip
!yes A | unzip modelos.zip

Archive:  faces.zip
   creating: faces/
   creating: faces/celular/
  inflating: faces/celular/0.png     
  inflating: faces/celular/1.jpeg    
   creating: faces/mask/
  inflating: faces/mask/0.png        
   creating: faces/monitor/
  inflating: faces/monitor/0.jpeg    
  inflating: faces/monitor/1.jpeg    
   creating: faces/print/
  inflating: faces/print/0.png       
   creating: faces/real/
  inflating: faces/real/0.jpeg       
   creating: faces/silicone/
  inflating: faces/silicone/0.png    
Archive:  modelos.zip
   creating: modelos/
  inflating: modelos/haarcascade_frontalface_default.xml  
  inflating: modelos/shape_predictor_5_face_landmarks.dat  
  inflating: modelos/shape_predictor_68_face_landmarks.dat  


### 2.2 Aplicação com Webcam (Tempo Real)

Para acessar a webcam do PC no google colab é precisar recorrer ao JavaScript. As funções abaixo mostra um exemplo de como fazer essa conexão

In [98]:
!wget -q https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx -O modelos/face_detection_yunet_2023mar.onnx

In [99]:
!wget -q https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx -O modelos/face_recognition_sface_2021dec.onnx

In [100]:
# Baixar o modelo (execute uma vez)
!wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task


In [101]:
%%capture
!pip uninstall mediapipe
!pip install -q --upgrade ultralytics transformers diffusers accelerate safetensors
!pip install -q supervision ipywidgets watermark
!pip install mediapipe==0.10.31


# Testes com LBP

Este caso não valida se o modelo aprendeu de fato, mas em testes de liveness o LBP teve pouca eficácia.
 Um dos motivos pode ser a base de dados escassa e a similaridade da textura facial nas imagens testadas.

In [102]:
import pandas as pd
import IPython.display as display
from google.colab import output
import cv2
import numpy as np
import base64
import io
import os
# Carregando as imagens e preparando os dados para treinamento
faces = []
labels = []
label_dict = {}

# Definindo o caminho para o diretório com imagens
df = pd.read_csv('liveness_train.csv')
print(df.head())
faces_folder_path = 'faces'
folders = [os.listdir(faces_folder_path + folder_name) for folder_name in df['file']]

for j, folder_el in enumerate(folders):
  for i, file_name in enumerate(folder_el):
    image_path = os.path.join(faces_folder_path + df['file'][j], file_name)
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is not None:
        image = cv2.resize(image, (640, 480), interpolation=cv2.INTER_AREA)
        faces.append(image)
        # Usando um label único para cada imagem (pode ser modificado para agrupar por pessoa)
        labels.append(j)
        label_dict[j] = df['type'][j]

faces = np.array(faces)
labels = np.array(labels)

# Treinando o reconhecedor
lbph_recognizer = cv2.face.LBPHFaceRecognizer_create()
lbph_recognizer.train(faces, labels)

def recognize_face(image):
    test_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if image is None:
        print("Erro: Imagem de teste não encontrada ou inválida.")
        return

    # Executando o reconhecimento
    label, confidence = lbph_recognizer.predict(test_img)

    # Encontrando o nome do label
    recognized_name = label_dict.get(label, "Desconhecido")

    # Exibindo os resultados
    print(f"Face reconhecida: {recognized_name} com confiança de {confidence}")

    return recognized_name, confidence

       file     type
0     /mask     mask
1  /monitor  monitor
2    /print    print
3  /celular  celular
4     /real     real


In [ ]:
from google.colab.patches import cv2_imshow
import cv2

images = [
    'faces/celular/0.png',
    'faces/mask/0.png',
    'faces/monitor/0.jpeg',
    'faces/print/0.png',
    'faces/real/0.jpeg',
    'faces/silicone/0.png',
]

for i, img in enumerate(images):
  res = cv2.imread(img)
  text, confidence = recognize_face(res)
  cv2.putText(res, text, (12, 30 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (30, 144, 255), 2)
  cv2_imshow(res)


# Liveness com modelo CLIP

Neste Experimento o CLIP se mostrou rápido e eficaz, na detecção de liveness. O CLIP pode reconhecer categorias definidas para seu uso, não categorizando o que não for especificado pelo sistema.

In [ ]:
import torch
import gc

gpu_ok = torch.cuda.is_available()

if gpu_ok:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU disponível: {props.name}")
    print(f"VRAM total: {props.total_memory / 1e9:.1f} GB")
else:
    print("!! NENHUMA GPU DETECTADA !!")
    print("Va em: Ambiente de execucao -> Alterar tipo de ambiente de execucao -> GPU (T4)")
    print("Sem GPU, as secoes de Stable Diffusion e SAM podem nao rodar em tempo habil de aula.")

DEVICE = torch.device("cuda" if gpu_ok else "cpu")
print(f"\nDevice em uso: {DEVICE}")


In [ ]:
import os
import urllib.request

os.makedirs("checkpoints", exist_ok=True)

print("Baixando CLIP...")
from transformers import CLIPModel, CLIPProcessor
_ = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
_ = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
del _
print("OK — CLIP\n")

gc.collect()
torch.cuda.empty_cache() if gpu_ok else None
print("Cache completo! Todos os checkpoints estao salvos localmente (~/.cache).")


In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image

clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
print("CLIP carregado.")

candidate_labels = [
    "person",
    "cellphone",
    "mask",
    "monitor",
    "silicone",
]

def _clip_infer_frame(frame_rgb):
    pil_frame = Image.fromarray(frame_rgb)
    inputs = clip_processor(text=candidate_labels, images=pil_frame, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        outputs = model_clip(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[0].cpu().numpy()
    ranking = sorted(zip(candidate_labels, probs), key=lambda x: -x[1])

    out = frame_rgb.copy()
    for i, (label, prob) in enumerate(ranking[:3]):
        text = f"{prob:.0%}  {label}"
        cv2.putText(out, text, (12, 30 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (30, 144, 255), 2)
    return out, ranking

# Testes de CLIP com imagens

Vemos aqui que o CLIP tem algumas classificações incorretas com imagens estaticas.
 Algumas classificações foram incorretas, a classificação de video, veremos a seguir.

In [ ]:
from google.colab.patches import cv2_imshow
import cv2
cv2_imshow(_clip_infer_frame(cv2.imread('faces/celular/0.png'))[0])
cv2_imshow(_clip_infer_frame(cv2.imread('faces/mask/0.png'))[0])
cv2_imshow(_clip_infer_frame(cv2.imread('faces/monitor/0.jpeg'))[0])
cv2_imshow(_clip_infer_frame(cv2.imread('faces/print/0.png'))[0])
cv2_imshow(_clip_infer_frame(cv2.imread('faces/real/0.jpeg'))[0])
cv2_imshow(_clip_infer_frame(cv2.imread('faces/silicone/0.png'))[0])

# Tirar a foto de análise

 Para testar a detecção e reconhecimento, precisamos de uma foto do usuário a ser reconhecido, neste caso o colab deve ter interação com o usuário para tirar a foto. Lembrando que tem que aceitar o uso de camera. Podendo ter que ser re executado após esta liberação.

In [ ]:
import IPython.display as display
from google.colab import output
import cv2
import numpy as np
import base64
import io
import os

def take_photo(filename='imagens/foto.jpg', quality=0.8):
    """
    Ativa a câmera e captura uma foto quando o usuário clica no botão "Capturar Foto".

    Parâmetros:
    - filename (str): Caminho onde a imagem será salva.
    - quality (float): Qualidade da imagem entre 0 e 1.

    Retorna:
    - filename (str): Caminho do arquivo salvo.
    """

    js = """
    async function capturePhoto(quality) {
        return new Promise((resolve, reject) => {
            const div = document.createElement('div');
            document.body.appendChild(div);

            const video = document.createElement('video');
            video.style.width = '100%';
            video.style.maxWidth = '640px';
            video.style.height = 'auto';
            div.appendChild(video);

            navigator.mediaDevices.getUserMedia({ video: true }).then(stream => {
                video.srcObject = stream;
                video.play();

                const button = document.createElement('button');
                button.innerText = '📷 Capturar Foto';
                button.style.display = 'block';
                button.style.marginTop = '10px';
                button.style.padding = '10px';
                button.style.fontSize = '16px';
                button.style.cursor = 'pointer';
                button.style.backgroundColor = 'red';
                button.style.color = 'white';
                div.appendChild(button);

                button.onclick = () => {
                    const canvas = document.createElement('canvas');
                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    canvas.getContext('2d').drawImage(video, 0, 0);
                    stream.getTracks().forEach(track => track.stop());  // Para a captura de vídeo
                    div.remove();  // Remove elementos do DOM

                    let dataUrl = canvas.toDataURL('image/jpeg', quality);
                    resolve(dataUrl);  // Retorna a imagem capturada
                };
            }).catch(err => {
                div.remove();
                reject(err);
            });
        });
    }
    """

    try:
        # Garante que o diretório de destino existe
        os.makedirs(os.path.dirname(filename), exist_ok=True)

        display.display(display.Javascript(js))
        data_url = output.eval_js('capturePhoto({})'.format(quality))

        if not data_url or not isinstance(data_url, str) or ',' not in data_url:
            raise ValueError("Erro: Nenhuma imagem foi capturada. Tente novamente.")

        # Remove o prefixo 'data:image/jpeg;base64,' da string Base64
        encoded_data = data_url.split(',')[1]

        try:
            # Decodifica os dados Base64 em bytes com tratamento de padding
            # Adiciona padding se necessário
            padding = len(encoded_data) % 4
            if padding:
                encoded_data += '=' * (4 - padding)

            image_data = base64.b64decode(encoded_data)
        except Exception as e:
            raise ValueError(f"Erro na decodificação Base64: {str(e)}")

        # Verifica se a conversão gerou bytes válidos
        if not image_data or len(image_data) == 0:
            raise ValueError("Erro: Dados da imagem vazios após decodificação Base64.")

        # Converte os bytes para um array NumPy
        image_array = np.frombuffer(image_data, dtype=np.uint8)

        # Verifica se o array não está vazio antes de tentar decodificar com OpenCV
        if image_array.size == 0:
            raise ValueError("Erro: O array de imagem está vazio. Tente novamente.")

        # Decodifica a imagem no formato OpenCV
        image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

        # Verifica se a imagem foi corretamente carregada
        if image is None:
            raise ValueError("Erro: OpenCV falhou ao decodificar a imagem.")

        # Salva a imagem corretamente
        cv2.imwrite(filename, image)

        print(f"📷 Imagem salva em: {filename}")
        return filename
    except Exception as e:
        raise RuntimeError(f"Erro ao acessar a câmera: {str(e)}")


# Execução do código
from IPython.display import Image as ColabImage
try:
    filename = take_photo("imagens/foto.jpg")

    # Exibe a imagem salva
    display.display(ColabImage(filename))
except Exception as err:
    print("Erro:", str(err))

# De Imagem para real time

 Abaixo estão as funções para executar a detecção realtime

In [ ]:
import cv2
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

def setup_camera():
    js_code = """
    var old = document.getElementById('camera-container');
    if (old) old.remove();

    window._stopRequested = false;

    var div = document.createElement('div');
    div.id = 'camera-container';
    div.style.cssText = 'font-family: monospace; padding: 10px;';
    document.body.appendChild(div);

    div.innerHTML = `
        <h3 style="color:#4CAF50;">Detector de Faces — liveness </h3>
        <video id="webcam" autoplay playsinline style="display:none;"></video>
        <canvas id="canvas" style="display:block;"></canvas>
        <div style="position:relative; display:inline-block;">
            <img id="output" style="display:block; margin:auto; border:3px solid #4CAF50; border-radius:8px; max-width:640px;">
            <div id="face-counter" style="
                position:absolute; top:8px; left:8px;
                background:rgba(0,0,0,0.6); color:#4CAF50;
                padding:4px 10px; border-radius:4px; font-size:14px; font-weight:bold;">
                Faces: 0
            </div>
            <div id="fps-counter" style="
                position:absolute; top:8px; right:8px;
                background:rgba(0,0,0,0.6); color:#FFC107;
                padding:4px 10px; border-radius:4px; font-size:14px;">
                FPS: --
            </div>
        </div>

        <div style="margin-top:10px; display:flex; align-items:center; gap:12px;">
            <div id="status-bar" style="color:#888; font-size:12px;">
                Inicializando câmera...
            </div>

            <!-- BOTÃO DE PARAR -->
            <button id="stop-btn" onclick="
                window._stopRequested = true;
                this.disabled = true;
                this.innerText = 'Desativando ...';
                this.style.background = '#555';
            " style="
                background: #e53935;
                color: white;
                border: none;
                padding: 6px 16px;
                border-radius: 6px;
                font-size: 13px;
                font-weight: bold;
                cursor: pointer;
                display: flex;
                align-items: center;
                gap: 6px;
            ">
                &#9632; Desativar
            </button>
        </div>
    `;

    window._cameraStream = null;

    async function setupCamera() {
        const video = document.getElementById('webcam');
        const stream = await navigator.mediaDevices.getUserMedia({
            video: { width: 640, height: 480 },
            audio: false,
        });
        window._cameraStream = stream;
        video.srcObject = stream;
        return new Promise((resolve) => {
            video.onloadedmetadata = () => {
                document.getElementById('status-bar').innerHTML = 'Detecção de Faces ativa';
                resolve();
            };
        });
    }

    async function captureFrame() {
        const video = document.getElementById('webcam');
        const canvas = document.getElementById('canvas');
        const ctx = canvas.getContext('2d');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        return canvas.toDataURL('image/jpeg', 0.85);
    }

    window.captureFrame = captureFrame;
    setupCamera();
    """
    display(Javascript(js_code))
    time.sleep(3)


def stop_camera():
    js_stop = """
    if (window._cameraStream) {
        window._cameraStream.getTracks().forEach(track => track.stop());
        window._cameraStream = null;
    }
    var statusBar = document.getElementById('status-bar');
    if (statusBar) {
        statusBar.innerHTML = 'Câmera desligada.';
        statusBar.style.color = '#f44336';
    }
    var btn = document.getElementById('stop-btn');
    if (btn) {
        btn.innerText = 'Desativado';
        btn.style.background = '#388E3C';
    }
    """
    eval_js(js_stop)
    print("Câmera desligada.")


def should_stop():
    """
    Verifica se o botão foi clicado consultando a flag JS.
    Retorna True se _stopRequested === true no navegador.
    """
    try:
        return bool(eval_js('window._stopRequested === true'))
    except:
        return False


def clear_output_display():
    display(Javascript('google.colab.output.clearAll();'))
    time.sleep(1)


def capture_frame():
    js_result = eval_js('captureFrame()')
    if not js_result:
        return None
    return b64decode(js_result.split(',')[1])


def update_display(frame, n_faces=0, fps=0.0):
    _, buffer = cv2.imencode('.jpg', frame)
    frame_b64 = b64encode(buffer).decode('utf-8')
    eval_js(f"""
    document.getElementById('output').src = 'data:image/jpeg;base64,{frame_b64}';
    document.getElementById('face-counter').innerText = 'Faces: {n_faces}';
    document.getElementById('fps-counter').innerText = 'FPS: {fps:.1f}';
    """)

# Teste Básico com DeepFake

  Um Teste Prático com deepfake para futura análise

In [ ]:
import numpy as np
import mediapipe as mp
from skimage.exposure import match_histograms
import argparse
from moviepy.editor import VideoFileClip
import base64
import io
import os
import cv2
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

BaseOptions           = mp.tasks.BaseOptions
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode
model_path = "face_landmarker.task"

# ── Conexões completas (contornos + íris) ──────────────────────
FACEMESH_FACE_OVAL = frozenset([
    (10,338),(338,297),(297,332),(332,284),(284,251),(251,389),
    (389,356),(356,454),(454,323),(323,361),(361,288),(288,397),
    (397,365),(365,379),(379,378),(378,400),(400,377),(377,152),
    (152,148),(148,176),(176,149),(149,150),(150,136),(136,172),
    (172,58),(58,132),(132,93),(93,234),(234,127),(127,162),
    (162,21),(21,54),(54,103),(103,67),(67,109),(109,10),
])
FACEMESH_LEFT_EYE = frozenset([
    (263,249),(249,390),(390,373),(373,374),(374,380),(380,381),
    (381,382),(382,362),(263,466),(466,388),(388,387),(387,386),
    (386,385),(385,384),(384,398),(398,362),
])
FACEMESH_RIGHT_EYE = frozenset([
    (33,7),(7,163),(163,144),(144,145),(145,153),(153,154),
    (154,155),(155,133),(33,246),(246,161),(161,160),(160,159),
    (159,158),(158,157),(157,173),(173,133),
])
FACEMESH_LEFT_EYEBROW = frozenset([
    (276,283),(283,282),(282,295),(295,285),(300,293),
    (293,334),(334,296),(296,336),(276,300),(285,336),
])
FACEMESH_RIGHT_EYEBROW = frozenset([
    (46,53),(53,52),(52,65),(65,55),(70,63),
    (63,105),(105,66),(66,107),(46,70),(55,107),
])
FACEMESH_LIPS = frozenset([
    (61,146),(146,91),(91,181),(181,84),(84,17),(17,314),
    (314,405),(405,321),(321,375),(375,291),(61,185),(185,40),
    (40,39),(39,37),(37,0),(0,267),(267,269),(269,270),
    (270,409),(409,291),(78,95),(95,88),(88,178),(178,87),
    (87,14),(14,317),(317,402),(402,318),(318,324),(324,308),
    (78,191),(191,80),(80,81),(81,82),(82,13),(13,312),
    (312,311),(311,310),(310,415),(415,308),
])
FACEMESH_CONTOURS = (
    FACEMESH_LIPS | FACEMESH_LEFT_EYE | FACEMESH_LEFT_EYEBROW
    | FACEMESH_RIGHT_EYE | FACEMESH_RIGHT_EYEBROW | FACEMESH_FACE_OVAL
)
FACEMESH_IRISES = frozenset([
    (474,475),(475,476),(476,477),(477,474),
    (469,470),(470,471),(471,472),(472,469),
])

def get_landmarks(image) -> list[tuple[int, int]] | None:
    """
    Detecta os pontos de referência faciais em uma imagem.

    Args:
        image: A imagem de entrada (formato BGR).
        face_mesh: O objeto FaceMesh do MediaPipe.

    Returns:
        Uma lista de tuplas (x, y) dos pontos de referência ou None se nenhum rosto for encontrado.
    """
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.2,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )

    with FaceLandmarker.create_from_options(options) as landmarker:
        cv_image = img_rgb
        if cv_image is None:
            print("Image not found:")
            return

        rgb_image = cv_image
        mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
        result    = landmarker.detect(mp_image)
    if not result.face_landmarks:
        print("No faces detected.")
        return []

    # Assume o primeiro rosto detectado

    h, w = image.shape[:2]
    landmarks = [(int(lm.x * w), int(lm.y * h)) for lm in result.face_landmarks[0]]
    return landmarks

def delaunay_triangulation(rect: tuple[int, int, int, int], points: list[tuple[int, int]]) -> list[tuple[int, int, int]]:
    """
    Calcula a triangulação de Delaunay para um conjunto de pontos.
    Versão otimizada usando um dicionário para lookup de índice.
    """

    points_dict = {p: i for i, p in enumerate(points)}

    subdiv = cv2.Subdiv2D(rect)
    subdiv.insert(points)

    triangle_list = subdiv.getTriangleList()
    triangles = []

    for t in triangle_list:
        pts = [(int(t[0]), int(t[1])), (int(t[2]), int(t[3])), (int(t[4]), int(t[5]))]

        # Verifica se os pontos do triângulo estão dentro do retângulo
        if rect[0] <= pts[0][0] <= rect[2] and rect[1] <= pts[0][1] <= rect[3] and \
           rect[0] <= pts[1][0] <= rect[2] and rect[1] <= pts[1][1] <= rect[3] and \
           rect[0] <= pts[2][0] <= rect[2] and rect[1] <= pts[2][1] <= rect[3]:

            indices = [points_dict.get(p) for p in pts]

            if all(idx is not None for idx in indices):
                triangles.append(tuple(indices))

    return triangles

def warp_triangle(src_img: np.ndarray, dst_img: np.ndarray, t_src: list, t_dst: list):
    """
    Deforma um triângulo da imagem de origem para a imagem de destino.
    """
    r_dst = cv2.boundingRect(np.float32([t_dst]))
    x, y, w, h = r_dst

    # Corta o triângulo de destino para processamento
    t_dst_rect = [(p[0] - x, p[1] - y) for p in t_dst]

    mask = np.zeros((h, w, 3), dtype=np.float32)
    cv2.fillConvexPoly(mask, np.int32(t_dst_rect), (1.0, 1.0, 1.0), 16, 0)

    # Deforma o triângulo da imagem de origem
    r_src = cv2.boundingRect(np.float32([t_src]))
    src_rect_img = src_img[r_src[1]:r_src[1]+r_src[3], r_src[0]:r_src[0]+r_src[2]]
    t_src_rect = [(p[0] - r_src[0], p[1] - r_src[1]) for p in t_src]

    # Matriz de transformação afim
    M = cv2.getAffineTransform(np.float32(t_src_rect), np.float32(t_dst_rect))
    warped_src_rect = cv2.warpAffine(src_rect_img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

    # Combina a área deformada com a imagem de destino
    dst_rect_area = dst_img[y:y+h, x:x+w]
    dst_rect_area = dst_rect_area * (1 - mask) + warped_src_rect * mask
    dst_img[y:y+h, x:x+w] = dst_rect_area

def elliptical_face_mask(size: tuple[int, int], feather: float = 0.20) -> np.ndarray:
    """Cria uma máscara elíptica suavizada para blending."""
    h, w = size
    mask = np.zeros((h, w), dtype=np.uint8)
    center = (w // 2, h // 2)
    axes = (int(w * 0.43), int(h * 0.57))
    cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)

    k_size = int(min(w, h) * feather)
    if k_size % 2 == 0: k_size += 1
    if k_size < 3: k_size = 3

    mask = cv2.GaussianBlur(mask, (k_size, k_size), 0)
    return mask

def create_comparison_view(original: np.ndarray, source: np.ndarray, result: np.ndarray, height: int) -> np.ndarray:
    """Cria uma visualização lado a lado para comparação."""
    def resize_and_label(img, text):
        h, w = img.shape[:2]
        scale = height / h
        resized = cv2.resize(img, (int(w * scale), height), interpolation=cv2.INTER_AREA)
        cv2.putText(resized, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        return resized

    original_vis = resize_and_label(original, "ORIGINAL")
    source_vis = resize_and_label(source, "SOURCE")
    result_vis = resize_and_label(result, "RESULT")

    return np.hstack([original_vis, source_vis, result_vis])

def draw_landmarks(image: np.ndarray, landmarks: list[tuple[int, int]], color=(0, 255, 255)) -> np.ndarray:
    """Desenha os pontos de referência faciais na imagem."""
    img_copy = image.copy()
    for (x, y) in landmarks:
        cv2.circle(img_copy, (x, y), 1, color, -1, lineType=cv2.LINE_AA)
    return img_copy

def create_comparison_view(original: np.ndarray, source: np.ndarray, result: np.ndarray, height: int,
                          show_landmarks=False, landmarks=None) -> np.ndarray:
    """Cria uma visualização lado a lado para comparação, desenhando landmarks se solicitado."""
    def resize_and_label(img, text):
        h, w = img.shape[:2]
        scale = height / h
        resized = cv2.resize(img, (int(w * scale), height), interpolation=cv2.INTER_AREA)
        cv2.putText(resized, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        return resized

    original_vis = original
    if show_landmarks and landmarks is not None:
        original_vis = draw_landmarks(original_vis, landmarks, color=(0,255,255))
    original_vis = resize_and_label(original_vis, "ORIGINAL")
    source_vis = resize_and_label(source, "SOURCE")
    result_vis = resize_and_label(result, "RESULT")
    return np.hstack([original_vis, source_vis, result_vis])

def run_face_swap(source_path: str, show_landmarks = True):
    """
    Executa o processo de face swap em tempo real a partir de uma câmera.
    """
    clear_output_display()
    setup_camera()

    print("Detecção facial iniciada.")
    print("Clique em [ ■ Desativar Camera ] na interface ou interrompa a célula.\n")


    src_img = cv2.resize(cv2.imread(source_path), (640, 480), interpolation=cv2.INTER_AREA)

    if src_img is None:
        print(f"Erro: Não foi possível encontrar a imagem de origem em '{source_path}'")
        return

    src_landmarks = get_landmarks(src_img)
    if src_landmarks is None:
        print("Erro: Nenhum rosto encontrado na imagem de origem.")
        return

    h, w = src_img.shape[:2]
    rect = (0, 0, w, h)
    triangles = delaunay_triangulation(rect, src_landmarks)

    display_height = 480

    frame_count = 0
    start_time = time.time()
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('output.mp4', fourcc, 1.0, (640, 480))

    try:
        while True:
            if should_stop():
                print("\nDesativado pelo botão da interface.")
                break

            frame_binary = capture_frame()
            if frame_binary is None:
                time.sleep(0.5)
                continue
            frame = cv2.imdecode(np.frombuffer(frame_binary, dtype=np.uint8), -1)
            dst_landmarks = get_landmarks(frame)
            output = frame.copy()

            if dst_landmarks:
                # 1. Warping Delaunay
                for tri_indices in triangles:
                    t_src = [src_landmarks[i] for i in tri_indices]
                    t_dst = [dst_landmarks[i] for i in tri_indices]
                    warp_triangle(src_img, output, t_src, t_dst)

                # 2. Correção de Cor e Blending
                hull_indices = cv2.convexHull(np.array(dst_landmarks), returnPoints=False)
                hull_points = np.array([dst_landmarks[i[0]] for i in hull_indices])

                bbox = cv2.boundingRect(hull_points)
                x, y, w_box, h_box = bbox

                # Garante que a ROI não esteja vazia
                if w_box > 0 and h_box > 0:
                    face_dst_roi = frame[y:y+h_box, x:x+w_box]
                    face_src_roi = output[y:y+h_box, x:x+w_box]

                    try:
                        # Correção de cor
                        face_src_roi = match_histograms(face_src_roi, face_dst_roi, channel_axis=-1).astype(np.uint8)
                        output[y:y+h_box, x:x+w_box] = face_src_roi
                    except ValueError:
                        # Ocorre se a ROI for muito pequena ou de cor sólida
                        pass

                    # 3. Blending final com máscara
                    face_mask = elliptical_face_mask((h_box, w_box), feather=0.18)
                    mask_total = np.zeros(frame.shape[:2], dtype=np.uint8)
                    mask_total[y:y+h_box, x:x+w_box] = face_mask

                    center = (x + w_box // 2, y + h_box // 2)
                    try:
                        # Usa seamlessClone para um blending suave
                        output = cv2.seamlessClone(output, frame, mask_total, center, cv2.MIXED_CLONE)
                    except cv2.error:
                        # Ocorre se o centro estiver muito perto da borda
                        pass

            processed_frame = output

            frame_count += 1
            elapsed = time.time() - start_time
            fps = frame_count / elapsed if elapsed > 0 else 0.0

            update_display(processed_frame, n_faces=1, fps=fps)
            time.sleep(0.05)

    except KeyboardInterrupt:
        print("\nLoop interrompido pelo usuário.")
    except Exception as e:
        print(f"\nErro inesperado: {e}")
    finally:
        stop_camera()
        out.release()
        cv2.destroyAllWindows()

run_face_swap('random-person.jpeg', True)


# Aplicação final

  Foram utilizados diversas técnicas detecção de faces.

  Para reconhecimento foi utilizado comparação de embeddings com similaridade de cosseno utilizando um limiar minimo para reconhecimento de face.
  
  Para Liveness foi realizada uma abordagem híbrida (ativo e passivo), utilizei o clip com as classes de possíveis spoofing de reconhecimento, e fiz uma votação com o mais frequente para definir se é real um falso como passivo, e como ativo o usuário tem que piscar por 3 segundos ou mais, a partir dai ele começa a verificar o liveness e reconhecimento.

  Caso o usuário passe por 15 segundos sem reconhecimento o sistema devolve falha de reconhecimento.
  

In [ ]:

from google.colab.patches import cv2_imshow
import numpy as np
import pandas as pd
import glob
from tqdm import tqdm
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import seaborn as sns
import math
import time
import IPython
from collections import Counter

from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js, register_callback
from base64 import b64decode, b64encode

# Scikit learn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn import preprocessing
from sklearn.svm import SVC

#Exibição na mesma tela do Jupyter
%matplotlib inline
import dlib
from scipy.spatial import distance as dist
from scipy import stats as st

sns.set_style("whitegrid", {'axes.grid' : False})
import cv2
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import dlib
import mediapipe as mp
import matplotlib.pyplot as plt
from moviepy.editor import VideoFileClip
from typing import Literal

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detect_faces(frame):
    if frame is None:
        return None, 0

    try:
      gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
      faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
      n_faces = len(faces)
      for i, (x, y, w, h) in enumerate(faces):
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, f"Face {i+1}", (x, y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
      return frame, n_faces
    except:
      return frame, 0

detector_yunet = cv2.FaceDetectorYN_create(
    model="modelos/face_detection_yunet_2023mar.onnx",
    config="",
    input_size=(320, 320),
    score_threshold=0.6,
    nms_threshold=0.3,
    top_k=5000
)

def detect_faces_yunet(frame):
    imagem_bgr = frame
    altura, largura = imagem_bgr.shape[:2]

    detector_yunet.setInputSize((largura, altura))
    _, rostos = detector_yunet.detect(imagem_bgr)

    imagem_anotada = imagem_bgr.copy()

    if rostos is not None:
        for rosto in rostos:
            x, y, w, h = rosto[0:4].astype(int)
            confianca = rosto[-1]

            cv2.rectangle(imagem_anotada, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(imagem_anotada, f"{confianca:.2f}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            marcos = rosto[4:14].reshape(5, 2).astype(int)
            for (mx, my) in marcos:
                cv2.circle(imagem_anotada, (mx, my), 2, (0, 0, 255), -1)
    else:
        print("Nenhum rosto detectado.")
    return imagem_anotada, len(rostos)

recognizer_sface = cv2.FaceRecognizerSF_create(
    model="modelos/face_recognition_sface_2021dec.onnx",
    config=""
)

def verificar_similaridade(image_1, image_2, threshold=0.363):

    embedding_1 = extrair_embedding(image_1)
    embedding_2 = extrair_embedding(image_2)

    if embedding_1 is not None and embedding_2 is not None:
        similaridade = recognizer_sface.match(embedding_1, embedding_2, cv2.FaceRecognizerSF_FR_COSINE)
        print(f"Similaridade (cosseno): {similaridade:.3f}")
        print("Mesma pessoa" if similaridade >= threshold else "Pessoas diferentes")
        return similaridade >= threshold
    else:
        print("Não foi possível detectar rosto em uma das imagens.")

def extrair_embedding(imagem):
    """
    Detecta o rosto principal de uma imagem (YuNet) e extrai seu embedding facial (SFace).

    Parâmetros:
    - caminho_imagem: caminho da imagem de entrada.

    Retorna:
    - embedding: vetor de 128 dimensões representando o rosto.
    - Retorna None se nenhum rosto for detectado.
    """
    altura, largura = imagem.shape[:2]
    detector_yunet.setInputSize((largura, altura))

    _, rostos = detector_yunet.detect(imagem)
    if rostos is None:
        return None

    rosto_principal = rostos[0]  # assume o rosto de maior confiança como principal
    rosto_alinhado = recognizer_sface.alignCrop(imagem, rosto_principal)

    return recognizer_sface.feature(rosto_alinhado)


# Caminho do arquivo do modelo pretreinado para a detecção de 68 pontos de referência faciais
classificador_68_path = "modelos/shape_predictor_68_face_landmarks.dat"

# Caminho do arquivo do modelo pretreinado para a detecção de 5 pontos de referência faciais
classificador_5_path = "modelos/shape_predictor_5_face_landmarks.dat"

# Carrega o modelo do Dlib para detectar 68 pontos de referência no rosto
classificador_dlib_68 = dlib.shape_predictor(classificador_68_path)

# Carrega o modelo do Dlib para detectar 5 pontos de referência no rosto
classificador_dlib_5 = dlib.shape_predictor(classificador_5_path)

# Inicializa o detector de rostos do Dlib baseado em HOG + SVM
detector_face_dlib = dlib.get_frontal_face_detector()

FACE_COMPLETA = list(range(0, 68))
SOMBRANCELHA_DIREITA = list(range(17, 22))
SOMBRANCELHA_ESQUERDA = list(range(22, 27))
OLHO_DIREITO = list(range(36, 42))
OLHO_ESQUERDO = list(range(42, 48))
NARIZ = list(range(27, 35))
LABIO = list(range(48, 61))
LABIO_EXTERIOR = list(range(48, 61))
LABIO_INTERIOR = list(range(60, 68))
MANDIBULA = list(range(0, 17))

def obter_marcos(imagem, marcos_68_pontos=True):
    """
    Detecta marcos faciais (landmarks) em uma imagem usando o modelo do Dlib.

    Parâmetros:
    - imagem: imagem de entrada onde os rostos serão detectados.
    - marcos_68_pontos (bool): define se deve usar o modelo de 68 pontos (True) ou 5 pontos (False).

    Retorna:
    - Lista de matrizes contendo os marcos faciais de cada rosto detectado.
    - Retorna None se nenhum rosto for encontrado.
    """

    # Seleciona o classificador adequado com base na quantidade de pontos desejada
    classificador_dlib = classificador_dlib_68
    if not marcos_68_pontos:
        classificador_dlib = classificador_dlib_5

    # Detecta rostos na imagem usando o detector do Dlib (HOG + SVM)
    faces = detector_face_dlib(imagem, 1)  # O "1" indica que a imagem será redimensionada uma vez para melhorar a detecção

    # Se nenhum rosto for detectado, retorna None
    if len(faces) == 0:
        return []

    marcos_faciais = []  # Lista para armazenar os marcos faciais detectados

    # Itera sobre os rostos detectados
    for face in faces:
        shape = classificador_dlib(imagem, face)  # Obtém os marcos faciais para o rosto atual

        pontos = [] # Re-initialize pontos as a list for each face
        # Coleta as coordenadas dos pontos faciais
        num_points = 68 if marcos_68_pontos else 5
        for i in range(0, num_points):
            pontos.append([shape.part(i).x, shape.part(i).y])

        # Converte a lista de pontos para uma matriz NumPy
        pontos_matrix = np.matrix(pontos) # Convert to matrix after collecting all points for the face

        # Adiciona os marcos faciais à lista
        marcos_faciais.append(pontos_matrix) # Append the matrix to the list

    return marcos_faciais  # Retorna a lista com os marcos faciais dos rostos detectados

def anotar_marcos(imagem, marcos_faciais):
    """
    Desenha pontos dos marcos faciais na imagem.

    Parâmetros:
    - imagem: a imagem onde os marcos faciais serão desenhados.
    - marcos_faciais: lista de matrizes contendo os pontos dos marcos faciais.

    Retorna:
    - A imagem com os marcos faciais anotados.
    """

    # Verifica se há marcos faciais detectados
    if len(marcos_faciais) == 0:
        print("Não foi identificado nenhum marco facial.")
        return imagem  # Retorna a imagem original sem modificações

    # Percorre cada conjunto de marcos faciais detectados
    for marco_facial in marcos_faciais:
        for idx, ponto in enumerate(marco_facial):
            # Obtém a posição do ponto (x, y)
            centro = (ponto[0, 0], ponto[0, 1])

            # Desenha um círculo amarelo no ponto correspondente
            cv2.circle(imagem, centro, 3, (0, 255, 255), -1)  # -1 preenche o círculo

    return imagem  # Retorna a imagem com os marcos desenhados

def anotar_marcos_faciais_componente(imagem, marcos_faciais):
    """
    Desenha contornos ao redor de componentes faciais (lábios e olhos) usando os marcos faciais detectados.

    Parâmetros:
    - imagem: A imagem onde os contornos serão desenhados.
    - marcos_faciais: Lista contendo os marcos faciais detectados.

    Retorna:
    - A imagem com os contornos desenhados.
    """

    # Verifica se há marcos faciais detectados
    if len(marcos_faciais) == 0:
        print("Não foi identificado nenhum marco facial.")
        return imagem  # Retorna a imagem original sem alterações

    # Percorre cada conjunto de marcos faciais (um para cada rosto detectado)
    for marco_facial in marcos_faciais:
        for d in marco_facial:  # Parece redundante, pode ser removido

            # Obtém os pontos convexos ao redor dos lábios e desenha o contorno em verde
            pontos = cv2.convexHull(marco_facial[LABIO])
            cv2.drawContours(imagem, [pontos], 0, (0, 255, 0), 2)

            # Obtém os pontos convexos ao redor do olho esquerdo e desenha o contorno em verde
            pontos = cv2.convexHull(marco_facial[OLHO_ESQUERDO])
            cv2.drawContours(imagem, [pontos], 0, (0, 255, 0), 2)

            # Obtém os pontos convexos ao redor do olho direito e desenha o contorno em verde
            pontos = cv2.convexHull(marco_facial[OLHO_DIREITO])
            cv2.drawContours(imagem, [pontos], 0, (0, 255, 0), 2)

    return imagem  # Retorna a imagem com os contornos desenhados

def detect_faces_dlib(frame):
    # Obtém os marcos faciais da imagem
    marcos_faciais = obter_marcos(frame)
    # Desenha os marcos faciais na imagem
    imagem_marcos = anotar_marcos(frame, marcos_faciais)
    imagem_marcos = anotar_marcos_faciais_componente(imagem_marcos, marcos_faciais)

    return imagem_marcos, len(imagem_marcos)

def ear(pontos_olho):
    """
    Calcula a Razão de Aspecto do Olho (EAR - Eye Aspect Ratio), que é usada para detectar fechamento dos olhos.

    Parâmetros:
    - pontos_olho: Lista de pontos que representam os marcos faciais do olho.

    Retorna:
    - medida_ear: Um valor que representa a abertura do olho. Valores menores indicam que o olho está fechado.
    """

    if len(pontos_olho) == 0: return 1

    # Converter os pontos do olho em vetores 1-D para facilitar o cálculo
    ponto1 = np.array([pontos_olho[1][0, 0], pontos_olho[1][0, 1]])  # Canto interno superior
    ponto2 = np.array([pontos_olho[2][0, 0], pontos_olho[2][0, 1]])  # Superior do meio
    ponto3 = np.array([pontos_olho[3][0, 0], pontos_olho[3][0, 1]])  # Inferior do meio
    ponto4 = np.array([pontos_olho[4][0, 0], pontos_olho[4][0, 1]])  # Canto interno inferior
    ponto5 = np.array([pontos_olho[5][0, 0], pontos_olho[5][0, 1]])  # Canto externo inferior

    # Calcular as distâncias verticais entre os pontos do olho
    a = dist.euclidean(ponto2, ponto5)  # Distância entre a parte superior e inferior do meio do olho
    b = dist.euclidean(ponto3, ponto4)  # Outra distância vertical do olho
    c = dist.euclidean(ponto1, ponto4)  # Distância horizontal entre os cantos do olho

    # Calcular a Razão de Aspecto do Olho (EAR)
    medida_ear = (a + b) / (2.0 * c)  # Fórmula usada para detectar fechamento ocular

    return medida_ear  # Quanto menor o valor, mais fechado está o olho

def obter_pontos_olho(marcos_faciais, olho_indices):
    """
    Obtém os pontos dos olhos a partir dos marcos faciais detectados.

    Parâmetros:
    - marcos_faciais: Matriz contendo todos os marcos faciais detectados.
    - olho_indices: Lista de índices dos pontos específicos do olho.

    Retorna:
    - pontos_olho: Lista contendo os pontos do olho desejado.
    """

    pontos_olho = []  # Lista para armazenar os pontos do olho

    # Percorre os índices dos marcos faciais e adiciona os pontos correspondentes à lista
    for idx in olho_indices:
        if len(marcos_faciais) > 0: pontos_olho.append(marcos_faciais[0][idx])

    return pontos_olho  # Retorna os pontos do olho selecionado


def detect_blink(frame, threshold=0.8):
  marcos_faciais = obter_marcos(frame, marcos_68_pontos=True)
  pontos_olho_direito = obter_pontos_olho(marcos_faciais, OLHO_DIREITO)
  ear_olho_direito = ear(pontos_olho_direito)
  print("EAR do olho direito " + str(ear_olho_direito))

  pontos_olho_esquerdo = obter_pontos_olho(marcos_faciais, OLHO_ESQUERDO)
  ear_olho_esquerdo = ear(pontos_olho_esquerdo)
  print("EAR do olho esquerdo " + str(ear_olho_esquerdo))

  return ear_olho_direito < threshold or ear_olho_esquerdo < threshold


def extrair_olho(shape, eye_indices):
    """
    Extrai os pontos faciais correspondentes a um olho.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.
    - eye_indices: Lista de índices dos pontos faciais que representam o olho.

    Retorna:
    - Lista de pontos do olho.
    """
    points = map(lambda i: shape.part(i), eye_indices)
    return list(points)

def extrair_olho_centro(shape, eye_indices):
    """
    Calcula o ponto central de um olho baseado nos seus marcos faciais.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.
    - eye_indices: Lista de índices dos pontos faciais do olho.

    Retorna:
    - Tupla (x, y) representando o centro do olho.
    """
    points = extrair_olho(shape, eye_indices)
    xs = map(lambda p: p.x, points)  # Extrai coordenadas x dos pontos do olho
    ys = map(lambda p: p.y, points)  # Extrai coordenadas y dos pontos do olho
    return sum(xs) // 6, sum(ys) // 6  # Retorna a média das coordenadas como centro

def extrair_olho_centro_esquerdo(shape):
    """
    Obtém o centro do olho esquerdo.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.

    Retorna:
    - Tupla (x, y) representando o centro do olho esquerdo.
    """
    return extrair_olho_centro(shape, OLHO_ESQUERDO)

def extrair_olho_centro_esquerdo(shape):
    """
    Obtém o centro do olho esquerdo.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.

    Retorna:
    - Tupla (x, y) representando o centro do olho esquerdo.
    """
    return extrair_olho_centro(shape, OLHO_ESQUERDO)


def extrair_olho_centro_direito(shape):
    """
    Obtém o centro do olho direito.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.

    Retorna:
    - Tupla (x, y) representando o centro do olho direito.
    """
    return extrair_olho_centro(shape, OLHO_DIREITO)

def angulo_entre_pontos(p1, p2):
    """
    Calcula o ângulo de inclinação entre dois pontos.

    Parâmetros:
    - p1, p2: Tuplas (x, y) representando dois pontos no espaço.

    Retorna:
    - O ângulo em graus entre os dois pontos.
    """
    x1, y1 = p1
    x2, y2 = p2
    tan = (y2 - y1) / (x2 - x1)  # Calcula a tangente do ângulo
    return np.degrees(np.arctan(tan))  # Converte o ângulo para graus

def get_rotation_matrix(p1, p2):
    """
    Calcula a matriz de rotação para alinhar a face.

    Parâmetros:
    - p1: Tupla (x, y) do olho esquerdo.
    - p2: Tupla (x, y) do olho direito.

    Retorna:
    - Matriz de rotação do OpenCV para alinhamento.
    """
    angle = angulo_entre_pontos(p1, p2)  # Calcula o ângulo de inclinação da face
    x1, y1 = p1
    x2, y2 = p2
    xc = (x1 + x2) // 2  # Calcula o ponto médio entre os olhos
    yc = (y1 + y2) // 2
    M = cv2.getRotationMatrix2D((xc, yc), angle, 1)  # Obtém a matriz de rotação
    return M

def align_face(frame, tamanho=None, return_face=False):
    """
    Alinha a face em uma imagem com base na posição dos olhos.

    Parâmetros:
    - image_path: Caminho da imagem de entrada.
    - tamanho: Tupla opcional (largura, altura) para redimensionamento.
    - return_face: Se True, retorna a face alinhada em vez de exibir.

    Retorna:
    - Se `return_face` for True, retorna a imagem da face alinhada.
    - Caso contrário, exibe a imagem original e alinhada.
    """
    imagem = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    height, width = imagem.shape[:2]
    dets = detector_face_dlib(imagem, 1)

    if len(dets) == 0:
        return None

    for i, det in enumerate(dets):

        x = det.left()
        y = det.top()
        w = det.right() - x
        h = det.bottom() - y

        # Garante que as coordenadas não fiquem fora dos limites da imagem
        if x < 0: x = 0
        if y < 0: y = 0

        roi = imagem[y:y+h, x:x+w]  # Recorta a região do rosto detectado

        # Obtém os marcos faciais do rosto detectado
        shape = classificador_dlib_68(imagem, det)
        left_eye = extrair_olho_centro_esquerdo(shape)  # Centro do olho esquerdo
        right_eye = extrair_olho_centro_direito(shape)  # Centro do olho direito

        M = get_rotation_matrix(left_eye, right_eye)  # Matriz de rotação

        # Define tamanho personalizado se especificado
        if tamanho:
            width = tamanho[0]
            height = tamanho[1]

        # Aplica a rotação na imagem
        rotated = cv2.warpAffine(imagem, M, (width, height))

        # Recorta novamente a região do rosto após a rotação
        cropped = rotated[y:y+h, x:x+w]

    if return_face:
        return cropped

    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(roi)

    plt.subplot(1,2,2)
    plt.title("Alinhado")
    plt.imshow(cropped)

    plt.show()

# !wget -O face_landmarker.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

BaseOptions           = mp.tasks.BaseOptions
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

# ── Conexões completas (contornos + íris) ──────────────────────
FACEMESH_FACE_OVAL = frozenset([
    (10,338),(338,297),(297,332),(332,284),(284,251),(251,389),
    (389,356),(356,454),(454,323),(323,361),(361,288),(288,397),
    (397,365),(365,379),(379,378),(378,400),(400,377),(377,152),
    (152,148),(148,176),(176,149),(149,150),(150,136),(136,172),
    (172,58),(58,132),(132,93),(93,234),(234,127),(127,162),
    (162,21),(21,54),(54,103),(103,67),(67,109),(109,10),
])
FACEMESH_LEFT_EYE = frozenset([
    (263,249),(249,390),(390,373),(373,374),(374,380),(380,381),
    (381,382),(382,362),(263,466),(466,388),(388,387),(387,386),
    (386,385),(385,384),(384,398),(398,362),
])
FACEMESH_RIGHT_EYE = frozenset([
    (33,7),(7,163),(163,144),(144,145),(145,153),(153,154),
    (154,155),(155,133),(33,246),(246,161),(161,160),(160,159),
    (159,158),(158,157),(157,173),(173,133),
])
FACEMESH_LEFT_EYEBROW = frozenset([
    (276,283),(283,282),(282,295),(295,285),(300,293),
    (293,334),(334,296),(296,336),(276,300),(285,336),
])
FACEMESH_RIGHT_EYEBROW = frozenset([
    (46,53),(53,52),(52,65),(65,55),(70,63),
    (63,105),(105,66),(66,107),(46,70),(55,107),
])
FACEMESH_LIPS = frozenset([
    (61,146),(146,91),(91,181),(181,84),(84,17),(17,314),
    (314,405),(405,321),(321,375),(375,291),(61,185),(185,40),
    (40,39),(39,37),(37,0),(0,267),(267,269),(269,270),
    (270,409),(409,291),(78,95),(95,88),(88,178),(178,87),
    (87,14),(14,317),(317,402),(402,318),(318,324),(324,308),
    (78,191),(191,80),(80,81),(81,82),(82,13),(13,312),
    (312,311),(311,310),(310,415),(415,308),
])
FACEMESH_CONTOURS = (
    FACEMESH_LIPS | FACEMESH_LEFT_EYE | FACEMESH_LEFT_EYEBROW
    | FACEMESH_RIGHT_EYE | FACEMESH_RIGHT_EYEBROW | FACEMESH_FACE_OVAL
)
FACEMESH_IRISES = frozenset([
    (474,475),(475,476),(476,477),(477,474),
    (469,470),(470,471),(471,472),(472,469),
])

model_path = "face_landmarker.task"
img_path   = "/content/fiap-ml-visao-computacional/aula-4-analise-facial/imagens/mulher.jpg"


def draw_tesselation(annotated, pts, w, h):
    """Triangulação de Delaunay sobre os 468 landmarks — equivalente ao FACEMESH_TESSELATION."""
    subdiv = cv2.Subdiv2D((0, 0, w, h))
    for (px, py) in pts[:468]:
        # Subdiv2D exige floats e pontos dentro do rect
        if 0 < px < w - 1 and 0 < py < h - 1:
            subdiv.insert((float(px), float(py)))

    for t in subdiv.getTriangleList().astype(np.int32):
        p1, p2, p3 = (t[0], t[1]), (t[2], t[3]), (t[4], t[5])
        if all(0 <= p[0] < w and 0 <= p[1] < h for p in (p1, p2, p3)):
            cv2.line(annotated, p1, p2, (180, 180, 180), 1, cv2.LINE_AA)
            cv2.line(annotated, p2, p3, (180, 180, 180), 1, cv2.LINE_AA)
            cv2.line(annotated, p3, p1, (180, 180, 180), 1, cv2.LINE_AA)


def draw_landmarks_on_image(rgb_image, detection_result):
    annotated = np.copy(rgb_image)
    h, w = annotated.shape[:2]
    print(detection_result)
    for face_landmarks in detection_result.face_landmarks:
        pts = [to_pixel(lm.x, lm.y, w, h) for lm in face_landmarks]
        n   = len(pts)

        # Malha triangular (tesselação)
        draw_tesselation(annotated, pts, w, h)

        # Contornos — ciano
        for a, b in FACEMESH_CONTOURS:
            if a < n and b < n:
                cv2.line(annotated, pts[a], pts[b], (0, 220, 220), 1, cv2.LINE_AA)

        # Íris — verde
        for a, b in FACEMESH_IRISES:
            if a < n and b < n:
                cv2.line(annotated, pts[a], pts[b], (48, 255, 48), 1, cv2.LINE_AA)

    return annotated


def run_face_landmarker_on_image(frame):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.2,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )

    with FaceLandmarker.create_from_options(options) as landmarker:
        cv_image = frame
        if cv_image is None:
            print("Image not found:")
            return

        rgb_image = cv_image
        mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
        result    = landmarker.detect(mp_image)

    if not result.face_landmarks:
        print("No faces detected.")
        return frame, 1

    annotated = draw_landmarks_on_image(rgb_image, result)

    return annotated, 1

BaseOptions           = mp.tasks.BaseOptions
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

# ── Conexões (mesmas do código anterior) ──────────────────────
FACEMESH_FACE_OVAL = frozenset([
    (10,338),(338,297),(297,332),(332,284),(284,251),(251,389),
    (389,356),(356,454),(454,323),(323,361),(361,288),(288,397),
    (397,365),(365,379),(379,378),(378,400),(400,377),(377,152),
    (152,148),(148,176),(176,149),(149,150),(150,136),(136,172),
    (172,58),(58,132),(132,93),(93,234),(234,127),(127,162),
    (162,21),(21,54),(54,103),(103,67),(67,109),(109,10),
])
FACEMESH_LEFT_EYE = frozenset([
    (263,249),(249,390),(390,373),(373,374),(374,380),(380,381),
    (381,382),(382,362),(263,466),(466,388),(388,387),(387,386),
    (386,385),(385,384),(384,398),(398,362),
])
FACEMESH_RIGHT_EYE = frozenset([
    (33,7),(7,163),(163,144),(144,145),(145,153),(153,154),
    (154,155),(155,133),(33,246),(246,161),(161,160),(160,159),
    (159,158),(158,157),(157,173),(173,133),
])
FACEMESH_LEFT_EYEBROW = frozenset([
    (276,283),(283,282),(282,295),(295,285),(300,293),
    (293,334),(334,296),(296,336),(276,300),(285,336),
])
FACEMESH_RIGHT_EYEBROW = frozenset([
    (46,53),(53,52),(52,65),(65,55),(70,63),
    (63,105),(105,66),(66,107),(46,70),(55,107),
])
FACEMESH_LIPS = frozenset([
    (61,146),(146,91),(91,181),(181,84),(84,17),(17,314),
    (314,405),(405,321),(321,375),(375,291),(61,185),(185,40),
    (40,39),(39,37),(37,0),(0,267),(267,269),(269,270),
    (270,409),(409,291),(78,95),(95,88),(88,178),(178,87),
    (87,14),(14,317),(317,402),(402,318),(318,324),(324,308),
    (78,191),(191,80),(80,81),(81,82),(82,13),(13,312),
    (312,311),(311,310),(310,415),(415,308),
])
FACEMESH_CONTOURS = (
    FACEMESH_LIPS | FACEMESH_LEFT_EYE | FACEMESH_LEFT_EYEBROW
    | FACEMESH_RIGHT_EYE | FACEMESH_RIGHT_EYEBROW | FACEMESH_FACE_OVAL
)
FACEMESH_IRISES = frozenset([
    (474,475),(475,476),(476,477),(477,474),
    (469,470),(470,471),(471,472),(472,469),
])

model_path  = "face_landmarker.task"
output_path = "/content/jn_annotated.mp4"

def to_pixel(x_norm, y_norm, w, h):
    return int(min(max(x_norm, 0.0), 1.0) * w), int(min(max(y_norm, 0.0), 1.0) * h)

def draw_tesselation(annotated, pts, w, h):
    subdiv = cv2.Subdiv2D((0, 0, w, h))
    for (px, py) in pts[:468]:
        if 0 < px < w - 1 and 0 < py < h - 1:
            subdiv.insert((float(px), float(py)))
    for t in subdiv.getTriangleList().astype(np.int32):
        p1, p2, p3 = (t[0], t[1]), (t[2], t[3]), (t[4], t[5])
        if all(0 <= p[0] < w and 0 <= p[1] < h for p in (p1, p2, p3)):
            cv2.line(annotated, p1, p2, (180, 180, 180), 1, cv2.LINE_AA)
            cv2.line(annotated, p2, p3, (180, 180, 180), 1, cv2.LINE_AA)
            cv2.line(annotated, p3, p1, (180, 180, 180), 1, cv2.LINE_AA)


def draw_landmarks_on_frame(rgb_frame, detection_result):
    annotated = np.copy(rgb_frame)
    h, w = annotated.shape[:2]

    for face_landmarks in detection_result.face_landmarks:
        pts = [to_pixel(lm.x, lm.y, w, h) for lm in face_landmarks]
        n   = len(pts)

        draw_tesselation(annotated, pts, w, h)

        for a, b in FACEMESH_CONTOURS:
            if a < n and b < n:
                cv2.line(annotated, pts[a], pts[b], (0, 220, 220), 1, cv2.LINE_AA)

        for a, b in FACEMESH_IRISES:
            if a < n and b < n:
                cv2.line(annotated, pts[a], pts[b], (48, 255, 48), 1, cv2.LINE_AA)

    return annotated


# ── Processamento do vídeo ─────────────────────────────────────
def run_face_landmarker_on_video(file_name):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,   # modo vídeo: usa detect_for_video()
        num_faces=1,
        min_face_detection_confidence=0.2,
        min_face_presence_confidence=0.2,
        min_tracking_confidence=0.2,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )

    cap = cv2.VideoCapture(file_name)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    with FaceLandmarker.create_from_options(options) as landmarker:
        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb_frame        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image         = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            # detect_for_video exige timestamp em milissegundos
            timestamp_ms     = int(cap.get(cv2.CAP_PROP_POS_MSEC))
            result           = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.face_landmarks:
                annotated = draw_landmarks_on_frame(rgb_frame, result)
            else:
                annotated = rgb_frame

            writer.write(cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
            frame_idx += 1

            if frame_idx % 30 == 0:
                print(f"Processando... {frame_idx}/{total} frames")

    cap.release()
    writer.release()
    print("Concluído:", output_path)

    # ── Exibição (igual ao código original) ───────────────────────
    clip = VideoFileClip(output_path)
    clip.ipython_display(width=500)
def check_validation(frame, success: Literal["same", "different", "fail"]):
    img = ''
    if success == 'same':
      img = """..."""
    elif success == 'fail':
      img = """..."""

    if success == 'same' or success == 'fail':
        _, mask = cv2.threshold(cv2.resize(cv2.imdecode(np.frombuffer(b64decode(img.split(',')[1]), dtype=np.uint8), cv2.IMREAD_GRAYSCALE), (640, 480), interpolation=cv2.INTER_AREA), 64, 255, cv2.THRESH_BINARY_INV)
        frame = cv2.bitwise_and(frame, frame, mask=mask)

    return frame

def face_detection_loop(mode: Literal["haar", "yu_net", "dlib", "media_pipe"] = "media_pipe"):
    clear_output_display()
    setup_camera()

    print("Detecção facial iniciada.")
    print("Clique em [ ■ Desativar Camera ] na interface ou interrompa a célula.\n")

    frame_count = 0
    start_time = time.time()
    is_recording = False
    blinking_frames = 0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('output.mp4', fourcc, 1.0, (640, 480))
    terminate = False
    labels_acc = []
    try:
        while True:
            if should_stop():
                print("\nDesativado pelo botão da interface.")
                break

            frame_binary = capture_frame()
            if frame_binary is None:
                time.sleep(0.5)
                continue

            frame = cv2.imdecode(np.frombuffer(frame_binary, dtype=np.uint8), -1)
            height, width = frame.shape[:2]
            circle_center = (width // 2, height // 2)  # Coordenadas do círculo (centro do primeiro quadrante)
            circle_mask = cv2.ellipse(np.zeros_like(frame[:, :, 0], dtype="uint8"), circle_center, (width // 4, height // 2), 0, 0, 360, (255, 255, 255), -1)
            frame = cv2.bitwise_and(frame, frame, mask=circle_mask)
            if (mode == "haar"): processed_frame, n_faces = detect_faces(frame)
            elif (mode == "yu_net"): processed_frame, n_faces = detect_faces_yunet(frame)
            elif (mode == "dlib"): processed_frame, n_faces = detect_faces_dlib(frame)
            else: processed_frame, n_faces = run_face_landmarker_on_image(frame)

            if is_recording:
                cv2.circle(processed_frame, (50, 50), 25, (0, 0, 255), -1)
                out.write(frame)

            frame_count += 1
            elapsed = time.time() - start_time
            fps = frame_count / elapsed if elapsed > 0 else 0.0

            if detect_blink(frame, 0.85):
              blinking_frames+=1
            else:
              blinking_frames=0

            if(blinking_frames >= 3/fps):
              is_recording = True

            if is_recording:
              success = verificar_similaridade(frame, cv2.imread('imagens/foto.jpg'), 0.8)
              rec_frame, rank = _clip_infer_frame(frame)
              labels_acc.append(rank[0][0])
              c = Counter(labels_acc)
              label = c.most_common(1)[0][0]
              if success and len(labels_acc) >= 5/fps and label == 'person':
                processed_frame = check_validation(processed_frame, 'same')
                terminate = True
              else: processed_frame = check_validation(processed_frame, 'different')

            if(elapsed >= 15*fps):
                processed_frame = check_validation(processed_frame, 'fail')
                terminate = True

            update_display(processed_frame, n_faces=n_faces, fps=fps)
            time.sleep(0.05)

            if terminate: break

    except KeyboardInterrupt:
        print("\nLoop interrompido pelo usuário.")
    except Exception as e:
        print(f"\nErro inesperado: {e}")
    finally:
        stop_camera()
        out.release()
        cv2.destroyAllWindows()


# face_detection_loop('haar')
face_detection_loop('yu_net')
# face_detection_loop('dlib')
# face_detection_loop('media_pipe')
# face_detection_loop()


# Conclusão

O estudo de detecção e reconhecimento de faces com liveness, obteve um resultado satisfatório, podendo ser melhor explorado com mais dados e mais faces de pessoas diferentes.